In [2]:
import pandas as pd

In [7]:
import duckdb

conn = duckdb.connect("../data_pipeline/protein_stability_dbt/dev.duckdb")

print(conn.execute("SHOW TABLES").fetchall())

[]


In [8]:
tables = conn.execute("""
SELECT table_schema, table_name
FROM information_schema.tables
ORDER BY table_schema, table_name
""").fetchdf()

print(tables)

Empty DataFrame
Columns: [table_schema, table_name]
Index: []


In [3]:
## Datasets to explore
df_small = pd.read_csv('../data/raw/fireprot_upload/csvs/4_fireprotDB_bestpH.csv', index_col=0)
split_train = pd.read_csv('../data/raw/fireprot_upload/csvs/splits/fireprot_train.csv', index_col=0)
split_test = pd.read_csv('../data/raw/fireprot_upload/csvs/splits/fireprot_test.csv', index_col=0)
split_val = pd.read_csv('../data/raw/fireprot_upload/csvs/splits/fireprot_val.csv', index_col=0)
split_homologue_free = pd.read_csv('../data/raw/fireprot_upload/csvs/splits/fireprot_homologue_free.csv', index_col=0)

## Full fireprotDB dataset
df = pd.read_csv('../data/raw/fireprotdb_20251015-164116.csv', index_col=0)

In [4]:
df.head()

,SEQUENCE_ID,MUTANT_ID,SOURCE_SEQUENCE_ID,TARGET_SEQUENCE_ID,SEQUENCE_LENGTH,SUBSTITUTION,INSERTION,DELETION,PROTEIN,ORGANISM,...,CONSERVATION,WWPDB,B_FACTOR,IN_TUNNEL,IN_POCKET,PUBLICATION_PMID,PUBLICATION_DOI,PUBLICATION_YEAR,SOURCE_DATASET,REFERENCING_DATASET
EXPERIMENT_ID,,,,,,,,,,,,,,,,,,,,,


In [4]:
df

,experiment_id,protein_name,uniprot_id,pdb_id,chain,position,wild_type,mutation,ddG,dTm,...,hsw_job_id,datasets,sequence,pdb_id_corrected,dupe_detector,pdb_position,pdb_sequence,oligomeric_state,structure_method,resolution
23,LL000714,Immunoglobulin G-binding protein G,P06654,1PGA|1EM7|2GB1,A,1,M,A,-0.14,NaN,...,2i1nqi,NaN,MEKEKKVKYFLRKSAFGLASVSAAFLVGSTVFAVDSPIEDTPIIRN...,1PGA,1PGA-A-0-M-P06654,0,MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYD...,monomer,x-ray diffraction,NaN
24,LL000715,Immunoglobulin G-binding protein G,P06654,1PGA|1EM7|2GB1,A,1,M,D,-0.38,NaN,...,2i1nqi,NaN,MEKEKKVKYFLRKSAFGLASVSAAFLVGSTVFAVDSPIEDTPIIRN...,1PGA,1PGA-D-0-M-P06654,0,MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYD...,monomer,x-ray diffraction,NaN
25,LL000716,Immunoglobulin G-binding protein G,P06654,1PGA|1EM7|2GB1,A,1,M,E,-0.64,NaN,...,2i1nqi,NaN,MEKEKKVKYFLRKSAFGLASVSAAFLVGSTVFAVDSPIEDTPIIRN...,1PGA,1PGA-E-0-M-P06654,0,MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYD...,monomer,x-ray diffraction,NaN
26,LL000717,Immunoglobulin G-binding protein G,P06654,1PGA|1EM7|2GB1,A,1,M,F,-1.14,NaN,...,2i1nqi,NaN,MEKEKKVKYFLRKSAFGLASVSAAFLVGSTVFAVDSPIEDTPIIRN...,1PGA,1PGA-F-0-M-P06654,0,MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYD...,monomer,x-ray diffraction,NaN
27,LL000718,Immunoglobulin G-binding protein G,P06654,1PGA|1EM7|2GB1,A,1,M,G,-0.30,NaN,...,2i1nqi,NaN,MEKEKKVKYFLRKSAFGLASVSAAFLVGSTVFAVDSPIEDTPIIRN...,1PGA,1PGA-G-0-M-P06654,0,MTYKLILNGKTLKGETTTEAVDAATAEKVFKQYANDNGVDGEWTYD...,monomer,x-ray diffraction,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6604,PT025496,Ribose import binding protein RbsB,P02925,2DRI,A,213,A,C,NaN,-6.60,...,jungim,EASE-MM-238|ProTherm,MNMKKLATLVSAVALSATVSANAMAKDTIALVVSTLNNPFFVSLKD...,2DRI,2DRI-C-187-A-P02925,187,KDTIALVVSTLNNPFFVSLKDGAQKEADKLGYNLVVLDSQNNPAKE...,monomer,x-ray diffraction,NaN
6605,PT025629,Ovalbumin,P01012,1UHG,A,143,R,A,NaN,-3.30,...,ufkeo8,ProTherm,MGSIGAASMEFCFDVFKELKVHHANENIFYCPIAIMSALAMVYLGA...,1UHG,1UHG-A-141-R-P01012,141,GSIGAASMEFCFDVFKELKVHHANENIFYCPIAIMSALAMVYLGAK...,monomer,x-ray diffraction,NaN
6606,PT025712,Cocaine esterase,Q9L9D7,1JU3,A,172,T,R,NaN,3.78,...,uznt1b,ProTherm,MVDGNYSVASNVMVPMRDGVRLAVDLYRPDADGPVPVLLVRNPYDK...,1JU3,1JU3-R-167-T-Q9L9D7,167,NYSVASNVMVPMRDGVRLAVDLYRPDADGPVPVLLVRNPYDKFDVF...,monomer,x-ray diffraction,NaN
6607,PT025713,Cocaine esterase,Q9L9D7,1JU3,A,173,G,Q,NaN,2.97,...,uznt1b,ProTherm,MVDGNYSVASNVMVPMRDGVRLAVDLYRPDADGPVPVLLVRNPYDK...,1JU3,1JU3-Q-168-G-Q9L9D7,168,NYSVASNVMVPMRDGVRLAVDLYRPDADGPVPVLLVRNPYDKFDVF...,monomer,x-ray diffraction,NaN


In [5]:
splits = ['split_train', 'split_test', 'split_val', 'split_homologue_free']
for split in splits:
    print(f"{split}: {len(locals()[split])} samples")

split_train: 2686 samples
split_test: 350 samples
split_val: 402 samples
split_homologue_free: 2578 samples


In [6]:
## Working on the bestpH dataset first, to get a better understanding of the data.
df.info()

<class 'pandas.DataFrame'>
Index: 4997 entries, 23 to 6608
Data columns (total 42 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   experiment_id            4997 non-null   str    
 1   protein_name             4997 non-null   str    
 2   uniprot_id               4997 non-null   str    
 3   pdb_id                   4997 non-null   str    
 4   chain                    4997 non-null   str    
 5   position                 4997 non-null   int64  
 6   wild_type                4997 non-null   str    
 7   mutation                 4997 non-null   str    
 8   ddG                      3438 non-null   float64
 9   dTm                      1614 non-null   float64
 10  is_curated               4997 non-null   bool   
 11  type                     0 non-null      float64
 12  derived_type             0 non-null      float64
 13  interpro_families        4989 non-null   str    
 14  conservation             4309 non-null 

In [7]:
# Deletion of full null columns (type, derived_type etc.)
df = df.dropna(axis=1, how='all')

In [8]:
# Exploring some of the columns in more detail
df['asa'].value_counts()

asa
0.00     406
0.13     104
74.97     38
67.15     37
0.81      35
        ... 
90.50      1
49.34      1
49.38      1
14.31      1
73.92      1
Name: count, Length: 1243, dtype: int64

In [9]:
## Selection of only few columns for further exploration
selected_feature = ['pdb_id', 'wild_type', 'mutation', 'position', 'ddG']
df_selected = df[selected_feature]

In [10]:
df["wild_type"].value_counts()

wild_type
V    507
A    431
T    421
I    365
K    336
L    329
E    316
G    300
D    279
Y    252
S    219
N    213
R    184
F    181
P    132
H    121
W    120
M    114
Q    110
C     67
Name: count, dtype: int64

In [11]:
# uniprot ID's
df["uniprot_id"].value_counts()

uniprot_id
P00644    698
P06654    658
P00720    617
P61626    253
P00648    210
         ... 
P23360      1
P13487      1
P00183      1
Q7SIG1      1
P01012      1
Name: count, Length: 125, dtype: int64

## Lire le fichier fireprotdb.sql

In [11]:
from pathlib import Path
import duckdb

raw_dir = Path('../data/raw')
sql_files = sorted(raw_dir.glob('*.sql'))
print('SQL files trouvés:', len(sql_files))
for path in sql_files:
    print('-', path.name)

sql_dataframes = {}

for path in sql_files:
    print(f'\nTraitement: {path.name}')
    con = duckdb.connect(database=':memory:')
    try:
        current_stmt = []
        last_result = None
        with path.open('r', encoding='utf8', errors='ignore') as f:
            for line in f:
                stripped = line.rstrip('\n')
                if not stripped and not current_stmt:
                    continue
                current_stmt.append(stripped)
                if stripped.strip().endswith(';'):
                    sql = '\n'.join(current_stmt)
                    last_result = con.execute(sql)
                    current_stmt = []
        if current_stmt:
            sql = '\n'.join(current_stmt)
            last_result = con.execute(sql)

        file_data = {}
        if last_result is not None and last_result.description is not None:
            file_data['query_result'] = last_result.df()

        tables = con.execute('SHOW TABLES').df()
        for table_name in tables['name']:
            df_table = con.execute(f'SELECT * FROM "{table_name}"').df()
            file_data[table_name] = df_table

        sql_dataframes[path.stem] = file_data
        print(f"Chargé {path.name}: {len(file_data)} dataframe(s)")
    except Exception as exc:
        print(f"Erreur sur {path.name}: {exc}")
    finally:
        con.close()

if sql_dataframes:
    first_key = next(iter(sql_dataframes))
    print('\nPremier fichier chargé:', first_key)
    example = sql_dataframes[first_key]
    for name, df in example.items():
        print(f' - {name}: {len(df)} lignes, colonnes = {list(df.columns)}')
        display(df.head(3))


SQL files trouvés: 9
- 01_fireprotdb_2025-09-20.sql
- 03_search_index_view.sql
- 04_search_index_view_indexes.sql
- 05_search_view.sql
- 06_search_view_indexes.sql
- 07_sequence_view.sql
- 08_statistics_views.sql
- 09_integrity_check.sql
- fireprotdb.sql

Traitement: 01_fireprotdb_2025-09-20.sql
Erreur sur 01_fireprotdb_2025-09-20.sql: Catalog Error: unrecognized configuration parameter "statement_timeout"

Did you mean: "TimeZone"

Traitement: 03_search_index_view.sql
Erreur sur 03_search_index_view.sql: Not implemented Error: Cannot drop this type yet

Traitement: 04_search_index_view_indexes.sql
Erreur sur 04_search_index_view_indexes.sql: Not implemented Error: Creating partial indexes is not supported currently

Traitement: 05_search_view.sql
Erreur sur 05_search_view.sql: Not implemented Error: Cannot drop this type yet

Traitement: 06_search_view_indexes.sql
Erreur sur 06_search_view_indexes.sql: Not implemented Error: Creating partial indexes is not supported currently

Traitem

In [14]:
import duckdb
sql = open('../data/raw/fireprotdb.sql','r',encoding='utf8').read()
con = duckdb.connect('dev_copy.duckdb')
con.execute(sql)  # exécute le SQL (peut créer tables / insérer données)

ParserException: Parser Error: syntax error at or near "`"

LINE 17: CREATE TABLE IF NOT EXISTS `amino_acids_substitution` (
                                    ^